# Parking Detection

This notebook trains a YOLOv8n model to find empty parking slots, parked cars and illegally parked cars in car park camera images. Every detected car is then checked against the parking slot boundaries.

## Reminder

The parking slots are not found by the model. They must be defined once for each camera.

Before running Step 14:

1. Go to Step 14, "Define the parking slot boundaries".
2. Replace the `SLOTS` values with the 4 corner points (x, y) of every parking slot in your camera image. The current values only fit the demo image.
3. Run "Check for illegal parking" to see which cars overlap two slots or cross the slot boundary.

## Step 1: Install the libraries and load the dataset

On Colab, compress the `datasets` folder into `datasets.zip` and drag it into the Files panel on the left before running this cell. When running locally, keep this notebook next to the `datasets` folder.

In [ ]:
!pip install -q ultralytics

import os
import zipfile

# Local: keep this notebook next to the "datasets" folder.
# Colab: right-click the "datasets" folder, choose Compress to ZIP, then drag datasets.zip
#        into the Files panel on the left (or upload it when asked below).
if not os.path.exists("datasets"):
    zip_name = "datasets.zip"
    if not os.path.exists(zip_name):
        from google.colab import files
        zip_name = list(files.upload().keys())[0]
    with zipfile.ZipFile(zip_name) as z:
        z.extractall(".")

path = os.path.abspath(".")

## Step 2: Show the dataset path

In [ ]:
print("Dataset path:")
print(path)

### Show the dataset folders

In [ ]:
import os

print(os.listdir(path))

## Step 3: Find the dataset folder automatically

In [ ]:
DATASET_PATH = None

for root, dirs, files in os.walk(path):
    required_datasets = {"parking-camera-data", "illegal-parking-data-1", "illegal-parking-data-2"}

    if required_datasets.issubset(set(dirs)):
        DATASET_PATH = root
        break

print("Dataset path:")
print(DATASET_PATH)

## Step 4: Check the classes and number of images

The three datasets name their classes differently, so each one is mapped to the same three classes: Empty (free slot), Occupied (parked car) and Illegal (illegally parked car).

In [ ]:
import json
from collections import Counter

SPLITS = ["train", "valid", "test"]
class_names = ["Empty", "Occupied", "Illegal"]

# Each dataset names its classes differently, so we map them all to our 3 classes.
# For each dataset: COCO category id mapped to our class name (ids not listed are ignored)
# Note: illegal-parking-data-2 was exported with broken class names (Roboflow used lines of its readme as names).
# By drawing its boxes we found: category 2 = parked car (Occupied), category 3 = free slot (Empty),
# category 1 = only 8 random boxes (ignored).
SOURCES = {
    "parking-camera-data":    {1: "Empty", 3: "Occupied"},                   # 2 = LicensePlate (not used yet)
    "illegal-parking-data-1": {2: "Empty", 3: "Occupied", 1: "Illegal"},
    "illegal-parking-data-2": {3: "Empty", 2: "Occupied"},
}

coco = {}
for source in SOURCES:
    for split in SPLITS:
        with open(os.path.join(DATASET_PATH, source, split, "_annotations.coco.json")) as f:
            coco[source, split] = json.load(f)

print("Classes:")
print(class_names)

print("\nNumber of images and objects per dataset:")

for source in SOURCES:
    print(f"\n{source}")
    for split in SPLITS:
        counts = Counter(SOURCES[source].get(a["category_id"], "ignored") for a in coco[source, split]["annotations"])
        print(f"  {split}: {len(coco[source, split]['images'])} images, {dict(counts)}")

print("\nTotal number of classes:", len(class_names))

## Step 5: Display sample images from each class

Green boxes are empty slots, blue boxes are parked cars and red boxes are illegally parked cars.

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

COLORS = {"Empty": "lime", "Occupied": "dodgerblue", "Illegal": "red"}

# Find every image that contains each class
images_with_class = {name: [] for name in class_names}
for (source, split), data in coco.items():
    if split != "train":
        continue
    for img_info in data["images"]:
        names = {SOURCES[source].get(a["category_id"]) for a in data["annotations"] if a["image_id"] == img_info["id"]}
        for name in names & set(class_names):
            images_with_class[name].append((source, img_info))

plt.figure(figsize=(15, 12))

for row, class_name in enumerate(class_names):
    for col, (source, img_info) in enumerate(random.sample(images_with_class[class_name], 3)):
        ax = plt.subplot(3, 3, row * 3 + col + 1)
        ax.imshow(Image.open(os.path.join(DATASET_PATH, source, "train", img_info["file_name"])).convert("RGB"))

        # Only draw the boxes of this class
        for a in coco[source, "train"]["annotations"]:
            if a["image_id"] == img_info["id"] and SOURCES[source].get(a["category_id"]) == class_name:
                x, y, w, h = a["bbox"]
                ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, color=COLORS[class_name], linewidth=2))

        plt.title(class_name)
        plt.axis("off")

plt.tight_layout()
plt.show()

## Step 6: Load the dataset in YOLO format

YOLOv8 reads the images from folders, but every box must be written in its own format: one text file per image, with one line per object.

- Every image is resized to 512 x 512 pixels.
- Images are grouped into batches of 16.
- The data is already split into training, validation and test data, so the split is kept.

The three datasets are merged into one folder, `datasets/merged-yolo`. Illegal parking is rare, so training images that contain it are copied 8 times, which lets the model see them often enough to learn them.

In [ ]:
import shutil

# The merged dataset is saved in datasets/merged-yolo
MERGED_PATH = os.path.join(DATASET_PATH, "merged-yolo")

# Illegal parking is rare (~80 boxes vs. thousands of normal cars),
# so training images with illegal parking are copied 8 times, so the model sees them often enough
ILLEGAL_OVERSAMPLE = 8

if os.path.exists(MERGED_PATH):
    shutil.rmtree(MERGED_PATH)  # start fresh every time

for source, class_map in SOURCES.items():
    for split in SPLITS:
        data = coco[source, split]
        img_dir = os.path.join(MERGED_PATH, split, "images")
        lbl_dir = os.path.join(MERGED_PATH, split, "labels")
        os.makedirs(img_dir, exist_ok=True)
        os.makedirs(lbl_dir, exist_ok=True)

        for img_info in data["images"]:
            W, H = img_info["width"], img_info["height"]

            # YOLO format: one line per object: class_id x_center y_center width height (values 0-1)
            lines = []
            for a in data["annotations"]:
                if a["image_id"] != img_info["id"] or a["category_id"] not in class_map:
                    continue
                x, y, w, h = a["bbox"]
                class_id = class_names.index(class_map[a["category_id"]])
                lines.append(f"{class_id} {(x + w / 2) / W:.6f} {(y + h / 2) / H:.6f} {w / W:.6f} {h / H:.6f}")

            has_illegal = any(line.startswith("2 ") for line in lines)
            copies = ILLEGAL_OVERSAMPLE if (split == "train" and has_illegal) else 1

            base_name = f"{source}_{os.path.splitext(img_info['file_name'])[0]}"
            for k in range(copies):
                name = base_name if k == 0 else f"{base_name}_copy{k}"
                shutil.copy(os.path.join(DATASET_PATH, source, split, img_info["file_name"]),
                            os.path.join(img_dir, name + ".jpg"))
                with open(os.path.join(lbl_dir, name + ".txt"), "w") as f:
                    f.write("\n".join(lines))

for split in SPLITS:
    print(f"{split}: {len(os.listdir(os.path.join(MERGED_PATH, split, 'images')))} images")

## Step 7: Display one training batch

In [ ]:
import matplotlib.pyplot as plt

BATCH_SIZE = 16
train_img_dir = os.path.join(MERGED_PATH, "train", "images")
train_lbl_dir = os.path.join(MERGED_PATH, "train", "labels")

plt.figure(figsize=(10, 10))

for i, file_name in enumerate(random.sample(os.listdir(train_img_dir), 9)):
    ax = plt.subplot(3, 3, i + 1)
    image = Image.open(os.path.join(train_img_dir, file_name)).convert("RGB")
    W, H = image.size
    ax.imshow(image)

    with open(os.path.join(train_lbl_dir, os.path.splitext(file_name)[0] + ".txt")) as f:
        for line in f:
            if not line.strip():
                continue
            class_id, x, y, w, h = line.split()
            name = class_names[int(class_id)]
            x, y, w, h = float(x) * W, float(y) * H, float(w) * W, float(h) * H
            ax.add_patch(patches.Rectangle((x - w / 2, y - h / 2), w, h, fill=False, color=COLORS[name], linewidth=2))
    plt.axis("off")

plt.tight_layout()
plt.show()

## Step 8: Create the dataset file

`data.yaml` tells YOLO:

- `train`, `val` and `test`: where the training, validation and test images are
- `nc`: the number of classes
- `names`: the class names

In [ ]:
DATA_YAML = os.path.join(MERGED_PATH, "data.yaml")

with open(DATA_YAML, "w") as f:
    f.write(f"path: {MERGED_PATH}\n")
    f.write("train: train/images\nval: valid/images\ntest: test/images\n")
    f.write(f"nc: {len(class_names)}\n")
    f.write(f"names: {class_names}\n")

print(open(DATA_YAML).read())
print("Dataset pipeline is ready.")

## Step 9: Build the AI model (YOLOv8n)

This is the AI model that will detect the cars and parking slots in our images. Instead of creating a CNN from scratch, we use transfer learning: YOLOv8n was already trained on the COCO dataset, so it starts with useful knowledge of cars and objects.

The model has three parts. The backbone, pre-trained on COCO, takes the 512x512 input image and extracts features such as edges, wheels and car shapes. The neck combines small and large features so that cars both near and far from the camera are found. The detection head then predicts a box, a class (Empty, Occupied or Illegal) and a confidence score for every object.

Why YOLOv8n:

- Very fast
- Small model size
- Designed for edge devices
- High accuracy
- Compatible with TensorFlow Lite
- Can be compiled for the Coral Edge TPU

In [ ]:
from ultralytics import YOLO

# Load the pre-trained YOLOv8n model
model = YOLO("yolov8n.pt")

# Show model architecture
model.info()

## Step 10: Set the training settings

YOLO has no separate `compile()` step like TensorFlow. The optimizer, number of epochs and image size are chosen here and passed to training in the next step. The loss function and the metrics (precision, recall and mAP) are built in.

In [ ]:
import torch

# YOLO has no compile() step like TensorFlow, the settings are chosen here instead.
# The loss function (box loss + class loss) and the metrics (precision, recall, mAP) are built in.
TRAIN_SETTINGS = dict(
    data=DATA_YAML,
    epochs=60,
    patience=20,        # stop early if validation stops improving for 20 epochs
    imgsz=512,
    batch=BATCH_SIZE,
    optimizer="auto",   # YOLO picks the best optimizer
    device=0 if torch.cuda.is_available() else "cpu",   # use a GPU: on Colab choose Runtime, Change runtime type, T4 GPU
    project="runs",
    name="parking",
    exist_ok=True,
)

print("Model settings are ready.")

## Step 11: Train the AI model

In this step, the AI learns from the training images.

- Training dataset: used to teach the model.
- Validation dataset: used to evaluate the model after each epoch.
- Epochs = 60: the model sees the entire dataset up to 60 times.

During training, YOLO displays something like:

```text
Epoch 1/60
box_loss: 1.52
cls_loss: 2.10
Precision: 0.65
Recall: 0.58
mAP50: 0.61
```

Each epoch shows:

- box_loss: box position error (lower is better)
- cls_loss: class error (lower is better)
- Precision: how many detected boxes are correct
- Recall: how many real objects were found
- mAP50: detection accuracy (higher is better)

At each epoch, mAP50 should increase and the losses should decrease. If the final mAP50 is above 0.70, the model is performing well.

Use a GPU for training (on Colab: Runtime, Change runtime type, T4 GPU). On a CPU it is much slower.

In [ ]:
results = model.train(**TRAIN_SETTINGS)

RUN_DIR = str(results.save_dir)
print("Training results saved in:", RUN_DIR)

## Step 12: Plot training accuracy and loss

mAP50 is the detector's version of accuracy, so it is plotted together with precision and recall.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

history = pd.read_csv(os.path.join(RUN_DIR, "results.csv"))
history.columns = history.columns.str.strip()

# Accuracy values (mAP50 is the "accuracy" of a detector)
val_map50 = history["metrics/mAP50(B)"]
val_precision = history["metrics/precision(B)"]
val_recall = history["metrics/recall(B)"]

# Loss values
train_loss = history["train/box_loss"] + history["train/cls_loss"]
val_loss = history["val/box_loss"] + history["val/cls_loss"]

epochs_range = history["epoch"]

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, val_map50, label="Validation mAP50")
plt.plot(epochs_range, val_precision, label="Validation Precision")
plt.plot(epochs_range, val_recall, label="Validation Recall")
plt.title("Validation Accuracy (mAP50)")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, train_loss, label="Training Loss")
plt.plot(epochs_range, val_loss, label="Validation Loss")
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.legend()

plt.show()

## Step 13: Evaluate the model

This is the final performance of the AI model, measured on test images it has never seen before.

There are three important metrics:

- mAP50: how many objects were detected correctly. For example, mAP50 = 0.85 means the model correctly found about 85 out of every 100 objects. Higher is better.
- Precision: how many of the detected boxes are correct. Higher is better.
- Recall: how many of the real objects were found. Higher is better.

If the test mAP50 is high, the model can generalise well to new images.

In [ ]:
# Evaluate the model on the test dataset
best_model = YOLO(os.path.join(RUN_DIR, "weights", "best.pt"))
metrics = best_model.val(data=DATA_YAML, split="test", imgsz=512)

print("\n========== Model Evaluation ==========")
print(f"{'Class':<10}{'Precision':>10}{'Recall':>10}{'mAP50':>10}")
for i, name in enumerate(class_names):
    p, r, ap50, _ = metrics.box.class_result(i)
    print(f"{name:<10}{p:>10.3f}{r:>10.3f}{ap50:>10.3f}")
print(f"{'ALL':<10}{metrics.box.mp:>10.3f}{metrics.box.mr:>10.3f}{metrics.box.map50:>10.3f}")

## Step 14: Test the AI model on one image

This is the inference stage. The model is no longer learning; it is making predictions on a new image.

The input image is resized to 512x512 and passed through the YOLOv8n model, which returns a bounding box, a predicted class and a confidence score for every object it finds.

By default the demo car park image is used, because the parking slots below are drawn for it. To use your own image on Colab, set `UPLOAD_OWN_IMAGE = True` and redraw the slots for it.

In [ ]:
import glob

# False: use the demo car park image (the parking slots below are drawn for it)
# True:  upload your own image on Colab (then redraw the parking slots for it)
UPLOAD_OWN_IMAGE = False

if UPLOAD_OWN_IMAGE:
    from google.colab import files
    uploaded = files.upload()
    img_path = list(uploaded.keys())[0]
else:
    img_path = glob.glob(os.path.join(DATASET_PATH, "illegal-parking-data-1", "train", "360_F_140320034*.jpg"))[0]

print("Image:", img_path)

In [ ]:
CONF_THRESHOLD = 0.35

result = best_model.predict(img_path, conf=CONF_THRESHOLD, imgsz=512, verbose=False)[0]

plt.figure(figsize=(7, 7))
ax = plt.gca()
ax.imshow(Image.open(img_path).convert("RGB"))

counts = Counter()
for box in result.boxes:
    name = class_names[int(box.cls)]
    x1, y1, x2, y2 = box.xyxy[0].tolist()
    ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color=COLORS[name], linewidth=2))
    ax.text(x1, y1 - 4, f"{name} {float(box.conf) * 100:.2f}%", color=COLORS[name], fontsize=9, weight="bold")
    counts[name] += 1

plt.title(f"Cars: {counts['Occupied'] + counts['Illegal']} | Empty: {counts['Empty']} | Illegal: {counts['Illegal']}")
plt.axis("off")
plt.show()

for name in class_names:
    print(f"{name}: {counts[name]}")

### Define the parking slot boundaries

The camera does not move, so the parking slots are drawn only once. Each slot is 4 corner points (x, y) in the image.

In [ ]:
# Parking slots of the demo image (its white slot lines are at x = 18, 112, 208, 302, 398, 494)
SLOTS = {
    "P01": [(18, 120), (112, 120), (112, 445), (18, 445)],
    "P02": [(112, 120), (208, 120), (208, 445), (112, 445)],
    "P03": [(208, 120), (302, 120), (302, 445), (208, 445)],
    "P04": [(302, 120), (398, 120), (398, 445), (302, 445)],
    "P05": [(398, 120), (494, 120), (494, 445), (398, 445)],
}

plt.figure(figsize=(7, 7))
ax = plt.gca()
ax.imshow(Image.open(img_path).convert("RGB"))
for slot_id, corners in SLOTS.items():
    ax.add_patch(patches.Polygon(corners, fill=False, color="yellow", linewidth=2, linestyle="--"))
    ax.text(corners[0][0] + 5, corners[0][1] - 8, slot_id, color="yellow", fontsize=11, weight="bold")
plt.title("Parking slots")
plt.axis("off")
plt.show()

### Check for illegal parking

For every detected car, the code measures how much of the car lies inside each slot:

- At least 10% of the car in two or more slots: overlaps multiple slots (for example P02 + P03).
- At least 10% of the car outside all slots: crosses the parking boundary.
- Otherwise: properly parked.

A car is a violation if this rule flags it or the AI model classified it as Illegal. The penalty is simulated.

In [ ]:
import cv2
import numpy as np
from datetime import datetime

# A car counts as being in a slot if at least 10% of the car is inside it
OVERLAP_THRESHOLD = 0.10
PENALTY = "RM 50 (simulated summons)"


def overlap_area(polygon_a, polygon_b):
    area, _ = cv2.intersectConvexConvex(np.array(polygon_a, np.float32), np.array(polygon_b, np.float32))
    return area


def check_parking(car_box, slots):
    x1, y1, x2, y2 = car_box
    car_polygon = [(x1, y1), (x2, y1), (x2, y2), (x1, y2)]
    car_area = (x2 - x1) * (y2 - y1)

    share_in_slot = {slot_id: overlap_area(car_polygon, corners) / car_area for slot_id, corners in slots.items()}
    used_slots = [slot_id for slot_id, share in share_in_slot.items() if share >= OVERLAP_THRESHOLD]
    outside_share = 1 - sum(share_in_slot.values())

    if len(used_slots) >= 2:
        return used_slots, "Overlaps multiple slots"
    if outside_share >= OVERLAP_THRESHOLD:
        return used_slots, "Crosses parking boundary"
    return used_slots, "Properly parked"


detected_at = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

plt.figure(figsize=(8, 8))
ax = plt.gca()
ax.imshow(Image.open(img_path).convert("RGB"))
for slot_id, corners in SLOTS.items():
    ax.add_patch(patches.Polygon(corners, fill=False, color="yellow", linewidth=1.5, linestyle="--"))
    ax.text(corners[0][0] + 5, corners[0][1] - 8, slot_id, color="yellow", fontsize=10, weight="bold")

rows = []
for box in result.boxes:
    name = class_names[int(box.cls)]
    if name == "Empty":
        continue  # only cars are checked
    car_box = box.xyxy[0].tolist()
    used_slots, rule_status = check_parking(car_box, SLOTS)

    # Violation if the slot rule flags it OR the AI model classified it as Illegal
    violation = rule_status != "Properly parked" or name == "Illegal"
    reason = rule_status if rule_status != "Properly parked" else ("AI model: Illegal" if violation else "")
    vehicle = f"Car {len(rows) + 1:02d}"
    rows.append({
        "Vehicle": vehicle,
        "Parking Slots": " + ".join(used_slots) if used_slots else "-",
        "Status": "Parking Violation" if violation else "Properly Parked",
        "Reason": reason,
        "Detected At": detected_at,
        "Penalty": PENALTY if violation else "-",
    })

    x1, y1, x2, y2 = car_box
    color = "red" if violation else "lime"
    ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color=color, linewidth=3))
    ax.text(x1, y2 + 14, vehicle, color=color, fontsize=10, weight="bold")

plt.axis("off")
plt.show()

report = pd.DataFrame(rows, columns=["Vehicle", "Parking Slots", "Status", "Reason", "Detected At", "Penalty"])
report

## Step 15: Save the trained model

At this point training is finished. The model is saved so it can be used later without training again.

The `.pt` file contains:

- the model architecture
- the learned weights
- the class names

In [ ]:
# Save the trained model
shutil.copy(os.path.join(RUN_DIR, "weights", "best.pt"), "parking_yolov8n.pt")

print("Model saved successfully!")

## Step 16: Check that the model was saved

List the files in the current folder to make sure the model file was created.

In [ ]:
import os

print(os.listdir())

## Step 17: Measure the inference time

Run the saved model on the same image and measure how long one prediction takes.

In [ ]:
import time
import numpy as np

saved_model = YOLO("parking_yolov8n.pt")
print(f"Processing image: {img_path}")

saved_model.predict(img_path, imgsz=512, verbose=False)  # warm-up run (the first run is always slower)

start_time = time.perf_counter()
result = saved_model.predict(img_path, conf=CONF_THRESHOLD, imgsz=512, verbose=False)[0]
end_time = time.perf_counter()

inference_time = (end_time - start_time) * 1000

print("========== COLAB INFERENCE ==========")
print(f"Objects detected : {len(result.boxes)}")
print(f"Inference Time   : {inference_time:.2f} ms")

### More accurate timing measurement

A single measurement is not reliable. The same image is run 20 times and the average time is calculated.

In [ ]:
import time
import numpy as np

times = []

for _ in range(20):
    start_time = time.perf_counter()

    saved_model.predict(img_path, imgsz=512, verbose=False)

    end_time = time.perf_counter()

    times.append((end_time - start_time) * 1000)

average_time = np.mean(times)

print("========== COLAB INFERENCE RESULTS ==========")
print(f"Average Inference Time : {average_time:.2f} ms")
print(f"Fastest Time           : {np.min(times):.2f} ms")
print(f"Slowest Time           : {np.max(times):.2f} ms")
print(f"Frames Per Second      : {1000 / average_time:.1f} FPS")